In [1]:
!pip freeze | grep scikit-learn

scikit-learn==1.5.0


In [2]:
!python -V

Python 3.10.0


### Q1: Notebook

In [11]:
import pickle
import numpy as np
import pandas as pd

In [4]:
with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [5]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [6]:
df = read_data('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet')

In [7]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

In [8]:
y_pred

array([16.24590642, 26.1347962 , 11.88426424, ..., 11.59533603,
       13.11317847, 12.89999218])

#### What's the standard deviation of the predicted duration for this dataset?

In [12]:
np.std(y_pred)

6.247488852238703

### Q2: Preparing the output

In [23]:
# setting the required variables
year=2023
month=3
output_file = 'yellow_tripdata_2023-03_predictions.parquet'

In [17]:
df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

In [21]:
df_result = pd.DataFrame({'ride_id': df['ride_id'], 'predicted_duration': y_pred}) #Create results DataFrame

In [24]:
#Save dataframe as parquet:

df_result.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)

In [27]:
df_check = pd.read_parquet(output_file, engine='pyarrow')

print(df_check.shape)         # Should match (3316216, 2)
print(df_check.isnull().sum()) # Check for unexpected nulls

(3316216, 2)
ride_id               0
predicted_duration    0
dtype: int64


#### What's the size of the output file?

In [29]:
ls -lh

total 66M
-rw-r--r-- 1 sidd4ml sidd4ml  17K Jun 20 03:35 model.bin
-rw-r--r-- 1 sidd4ml sidd4ml 5.9K Jun 20 05:50 starter.ipynb
-rw-r--r-- 1 sidd4ml sidd4ml  66M Jun 20 05:46 yellow_tripdata_2023-03_predictions.parquet
